<a href="https://colab.research.google.com/github/Reito-alt/AI-Tuning/blob/main/KonversiModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install Library & Llama.cpp**

In [ ]:
!pip install -q -U transformers peft accelerate sentencepiece gguf
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
!pip install -q -r llama.cpp/requirements.txt

**Upload & Ekstrak Adapter**

In [ ]:
import os, zipfile
from google.colab import files

up = files.upload()
ZIP_NAME = list(up.keys())[0]
with zipfile.ZipFile(ZIP_NAME) as z:
  z.extractall(".")

ADAPTER_DIR = ZIP_NAME[:-4]
print("Folder Adapter:", ADAPTER_DIR)
print(os.listdir(ADAPTER_DIR))

**Merge Adapter**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_ID = "Qwen/Qwen2.5-1.5B-Instruct"  #harus sama dengan model yang dipakai training
MERGED_DIR = "qwen25-1.5b-asisten-belajar-merged"

base = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model = model.merge_and_unload()

model.save_pretrained(MERGED_DIR, safe_serialization=True)
AutoTokenizer.from_pretrained(ADAPTER_DIR).save_pretrained(MERGED_DIR)
print("Model merged tersimpan di:", MERGED_DIR)

**GGUF Converter**

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py qwen25-1.5b-asisten-belajar-merged \
    --outfile qwen25-1.5b-f16.gguf --outtype f16

**Build Llama Quantize**

In [ ]:
!cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF
!cmake --build llama.cpp/build --target llama-quantize -j 4

**Quantize Q4_K_M (4Bit)**

In [ ]:
!./llama.cpp/build/bin/llama-quantize qwen25-1.5b-f16.gguf qwen25-1.5b-q4_k_m.gguf Q4_K_M

**Download Result**

In [ ]:
from google.colab import files
files.download("qwen25-1.5b-q4_k_m.gguf")